# Assignment 2: Power-line insulator fault triage
## Training pipeline

**The decision.** A drone flies along a power line and photographs every insulator string. For each photo, someone has to decide:

| outcome | what happens |
|---|---|
| **PASS** | insulator left in service |
| **REVIEW** | an engineer looks at the photo and decides |
| **DISPATCH** | a repair crew is sent to the tower |

**Why the two errors cost different amounts.**
- **False negative** (a damaged insulator passed): it stays on the line until the next inspection. It can flash over and cause an outage, emergency repair or even a fire. Expensive.
- **False positive** (a healthy insulator, crew sent): one wasted crew trip. Much cheaper.

Because a miss costs far more than a false alarm, the best threshold will be **low**: we accept more false alarms to catch more faults.

**What this notebook does, top to bottom** (`uv run python main.py train` runs every cell):
1. Settings
2. Download the dataset from Roboflow Universe
3. Fix the split (the published one leaks and has no test set)
4. Describe the dataset
5. Finetune YOLO11n
6. Evaluate per class on the test split
7. Baselines: COCO zero-shot and "predict nothing"
8. Turn detections into one score per photo
9. Cost sweep → choose the thresholds
10. Save everything the dashboard needs

## 1. Settings

The training settings can be overridden with environment variables, e.g. `EPOCHS=5 uv run python main.py train`, so the same notebook runs on a slow CPU or a GPU.

| setting | default | why |
|---|---|---|
| `MODEL` | `yolo11n.pt` | the smallest YOLO11: fast enough for a CPU, and the brief allows n or s |
| `EPOCHS` | 20 | enough passes over ~770 training images to converge on a CPU budget |
| `IMGSZ` | 512 | images are resized to 512 px. Bigger helps small cracks but is slower |
| `BATCH` | 16 | images per gradient step |
| `SEED` | 0 | the same split and training every run (reproducible) |

In [ ]:
import json, os, platform, re, shutil, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

import decision                      # the cost maths, shared with the app (see section 9)
from api_key import load_api_key     # course helper: reads the key from .env

ROOT = Path.cwd()
RAW_DIR = ROOT / "datasets" / "insulators-v1-raw"     # exactly as downloaded
DATA_DIR = ROOT / "datasets" / "insulators-v1-split"   # after our grouped re-split
ART = ROOT / "artifacts"        # everything the dashboard reads
WEIGHTS = ROOT / "weights"      # the model we commit
RUNS = ROOT / "runs"            # Ultralytics scratch output (git-ignored)
ART.mkdir(exist_ok=True)

WORKSPACE, PROJECT, VERSION = "insulators-ljwyo", "insulators-c4gwk", 1

EPOCHS = int(os.environ.get("EPOCHS", 20))
IMGSZ = int(os.environ.get("IMGSZ", 512))
BATCH = int(os.environ.get("BATCH", 16))
MODEL = os.environ.get("MODEL", "yolo11n.pt")
SEED = 0
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if torch.cuda.is_available():
    DEVICE = "0"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
DEVICE = os.environ.get("DEVICE", DEVICE)
T0 = time.time()
print(f"device={DEVICE} epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH}")

## 2. Download the dataset

**Dataset:** [`insulators-ljwyo / insulators-c4gwk / v1`](https://universe.roboflow.com/insulators-ljwyo/insulators-c4gwk/dataset/1), CC BY 4.0, 1,099 drone images.

**Classes:** `insulator` (a whole insulator string), `defect`, `brokeninsulator`, `brokenpart`.

We download the public version **as-is**, not a fork, so anyone can re-download the same data. If the folder already exists, the download is skipped.

In [ ]:
if not (RAW_DIR / "data.yaml").exists():
    from roboflow import Roboflow
    rf = Roboflow(api_key=load_api_key())
    rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download(
        "yolov11", location=str(RAW_DIR), overwrite=True)

raw_cfg = yaml.safe_load((RAW_DIR / "data.yaml").read_text())
NAMES = list(raw_cfg["names"].values()) if isinstance(raw_cfg["names"], dict) else list(raw_cfg["names"])
for s in ("train", "valid", "test"):
    d = RAW_DIR / s / "images"
    print(f"published {s:5s}: {len(list(d.iterdir())) if d.exists() else 0} images")
print("classes:", NAMES)

## 3. Fix the split

There are two problems with the published version:

1. **No test split.** It ships 958 train / 141 valid / 0 test. We need a test set the model never saw to report honest numbers.
2. **Leakage.** The uploader pre-augmented the data: one original photo appears several times as flipped copies. The filenames give it away: `000000795_jpeg_jpg`, `000000795h_jpeg_jpg` (horizontal flip), `000000795v_jpeg_jpg` (vertical flip). If one copy is in train and another in test, the model has effectively **already seen the test photo**, and the score is inflated.

**Fix:** strip the suffixes to recover the original photo id, then split **by photo id** (70 / 15 / 15, seeded). All copies of a photo land in the same split.

This is a *local* re-split of the same public data. `artifacts/split_manifest.csv` records where every image went, so it is fully reproducible.

In [ ]:
def photo_id(filename: str) -> str:
    stem = filename.split(".rf.")[0]                              # drop Roboflow's '.rf.<hash>'
    stem = re.sub(r"(_(jpe?g|png))+$", "", stem, flags=re.I)      # drop '_jpeg_jpg', '_JPG'
    return re.sub(r"-?[hvd]$", "", stem, flags=re.I)              # drop the h / v / d copy suffix

# quick sanity check on real-looking names
for n in ["000000795_jpeg_jpg.rf.f49f.jpg", "000000795h_jpeg_jpg.rf.ada.jpg", "160605d_JPG.rf.x.jpg"]:
    print(f"{n:35s} -> {photo_id(n)}")

In [ ]:
pool = []   # (image path, published split)
for s in ("train", "valid", "test"):
    d = RAW_DIR / s / "images"
    if d.exists():
        pool += [(p, s) for p in sorted(d.iterdir()) if p.suffix.lower() in IMG_EXT]

ids = sorted({photo_id(p.name) for p, _ in pool})
rng = np.random.default_rng(SEED)
rng.shuffle(ids)
n = len(ids)
assign = {g: ("train" if i < 0.70 * n else "valid" if i < 0.85 * n else "test") for i, g in enumerate(ids)}

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
rows = []
for p, orig in pool:
    new = assign[photo_id(p.name)]
    for sub, src in (("images", p), ("labels", p.parent.parent / "labels" / (p.stem + ".txt"))):
        (DATA_DIR / new / sub).mkdir(parents=True, exist_ok=True)
        if src.exists():
            shutil.copy(src, DATA_DIR / new / sub / src.name)
    rows.append({"image": p.name, "photo_id": photo_id(p.name), "original_split": orig, "new_split": new})

manifest = pd.DataFrame(rows)
manifest.to_csv(ART / "split_manifest.csv", index=False)

leaking = int((manifest.groupby("photo_id")["original_split"].nunique() > 1).sum())
SPLIT_INFO = {
    "published_split": manifest["original_split"].value_counts().to_dict(),
    "new_split_images": manifest["new_split"].value_counts().to_dict(),
    "unique_photos": n,
    "images_per_photo_mean": round(len(manifest) / n, 2),
    "photos_leaking_across_published_train_valid": leaking,
}
print("[split]", SPLIT_INFO)

DATA_YAML = DATA_DIR / "data_abs.yaml"
DATA_YAML.write_text(yaml.safe_dump({"path": str(DATA_DIR), "train": "train/images", "val": "valid/images",
                                     "test": "test/images", "nc": len(NAMES), "names": NAMES}, sort_keys=False))

**How to read `[split]`:**
- `unique_photos` is the *real* number of independent photos. It's smaller than 1,099 because of the copies, so your effective sample size is smaller than it looks.
- `photos_leaking_across_published_train_valid` counts photos whose copies sat in both published train and valid. Anyone using the published split would have had inflated validation scores.

## 4. Describe the dataset

Computed from the label files, not copied from a screenshot. For each split we count:
- **instances per class**: how many boxes. A class with ~40 boxes in test gives a number you cannot trust.
- **images containing each class**
- **object size**: box area as a % of the image. Small objects (a chipped disc) are harder for a model at 512 px.

YOLO label format, one line per box: `class x_center y_center width height`, all as fractions of the image (0–1). So `width × height` = the share of the image the box covers.

In [ ]:
def images_in(split):
    d = DATA_DIR / split / "images"
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXT) if d.exists() else []

def read_labels(img):
    lab = img.parent.parent / "labels" / (img.stem + ".txt")
    if not lab.exists() or lab.stat().st_size == 0:
        return np.zeros((0, 5))
    rows = [list(map(float, l.split()[:5])) for l in lab.read_text().splitlines() if l.strip()]
    return np.array(rows) if rows else np.zeros((0, 5))

stats = {"source": f"{WORKSPACE}/{PROJECT}/{VERSION}",
         "url": f"https://universe.roboflow.com/{WORKSPACE}/{PROJECT}/dataset/{VERSION}",
         "classes": NAMES, "resplit": SPLIT_INFO, "splits": {}}
table = []
for split in ("train", "valid", "test"):
    imgs = images_in(split)
    inst, with_cls, areas, empty = {c: 0 for c in NAMES}, {c: 0 for c in NAMES}, {c: [] for c in NAMES}, 0
    for img in imgs:
        lab = read_labels(img)
        empty += int(len(lab) == 0)
        for k, c in enumerate(NAMES):
            sel = lab[lab[:, 0] == k] if len(lab) else lab
            inst[c] += len(sel); with_cls[c] += int(len(sel) > 0)
            areas[c] += list(sel[:, 3] * sel[:, 4]) if len(sel) else []
    size = {}
    for c in NAMES:
        a = np.array(areas[c])
        size[c] = {"median_area_pct": float(np.median(a) * 100), "share_below_1pct": float(np.mean(a < 0.01))} if len(a) else {}
        table.append({"split": split, "class": c, "instances": inst[c], "images_with_class": with_cls[c],
                      "median_box_%": round(size[c].get("median_area_pct", 0), 2)})
    stats["splits"][split] = {"images": len(imgs), "background_images": empty, "instances": inst,
                              "images_with_class": with_cls, "object_size": size}

(ART / "dataset_stats.json").write_text(json.dumps(stats, indent=2))
print("[data]", json.dumps({k: v["instances"] for k, v in stats["splits"].items()}))
print(pd.DataFrame(table).to_string(index=False))

## 5. Finetune YOLO11n

We start from `yolo11n.pt`, which has already learned general visual features on COCO (80 everyday object classes). **Finetuning** replaces its final layer with our 4 classes and keeps training on our images. That's why ~770 images are enough.

Ultralytics handles the training loop. Each epoch it also scores the validation split and keeps the best checkpoint (`best.pt`). We copy that into `weights/`, which is the file we commit and the file the app loads.

In [ ]:
model = YOLO(MODEL)
model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
            seed=SEED, deterministic=True, workers=2, project=str(RUNS), name="finetune",
            exist_ok=True, plots=True, verbose=False)

WEIGHTS.mkdir(exist_ok=True)
shutil.copy(RUNS / "finetune" / "weights" / "best.pt", WEIGHTS / "best.pt")
for f in ("results.csv", "results.png"):
    if (RUNS / "finetune" / f).exists():
        shutil.copy(RUNS / "finetune" / f, ART / f"train_{f}")
print("saved", WEIGHTS / "best.pt")

## 6. Evaluate on the test split, per class

The test split is the 15% of *photos* the model never saw, neither in training nor for picking the best epoch.

| metric | meaning |
|---|---|
| **Precision** | of the boxes the model drew for a class, the share that were right |
| **Recall** | of the real objects of a class, the share the model found |
| **AP50** | area under the precision-recall curve, where a box counts as correct if it overlaps the true box by ≥ 50% (IoU ≥ 0.5) |
| **mAP50-95** | the same, averaged over stricter overlap requirements (0.50 → 0.95). It rewards tight boxes |

The overall number is the average over classes, and **an average can hide a bad class**. For our decision, recall on the fault classes matters most, because missing a fault is the expensive error.

**Confusion matrix:** rows = predicted class, columns = true class, plus a "background" row and column. *True fault, predicted background* is a missed fault.

In [ ]:
m = YOLO(str(WEIGHTS / "best.pt")).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
                                        device=DEVICE, project=str(RUNS), name="test_eval",
                                        exist_ok=True, plots=True, verbose=False)
box = m.box
per_class = []
for i, c in enumerate(NAMES):
    idx = list(box.ap_class_index).index(i) if i in list(box.ap_class_index) else None
    g = (lambda arr: float(arr[idx]) if idx is not None else 0.0)
    per_class.append({"class": c, "precision": g(box.p), "recall": g(box.r), "AP50": g(box.ap50), "mAP50-95": g(box.ap)})

for f in ("confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png"):
    if (RUNS / "test_eval" / f).exists():
        shutil.copy(RUNS / "test_eval" / f, ART / f"test_{f}")

metrics = {"overall": {"precision": float(box.mp), "recall": float(box.mr), "AP50": float(box.map50), "mAP50-95": float(box.map)},
           "per_class": per_class,
           "confusion_matrix": {"labels": NAMES + ["background"], "matrix": m.confusion_matrix.matrix.astype(int).tolist(),
                                "orientation": "rows = predicted, cols = true"}}
print(pd.DataFrame(per_class).round(3).to_string(index=False))
print("overall:", {k: round(v, 3) for k, v in metrics["overall"].items()})

## 7. Baselines

The brief asks for two comparisons on the same test split:

1. **COCO model, zero-shot.** The original `yolo11n.pt`, no finetuning. COCO has no insulator or fault classes, so on detection metrics it scores **0 by construction**: it can never draw a correctly-labelled box. To give it a fair shot on the *decision*, we use its highest confidence for *any* object as a "something's there" score.
2. **Trivial: predict nothing.** It never flags anything, so every photo passes. AP = 0, recall = 0. On the decision, this is the same as having no inspection model at all.

If our finetuned model can't beat these, that's a finding worth reporting.

In [ ]:
metrics["baselines_detection"] = [
    {"model": "Finetuned yolo11n", **metrics["overall"]},
    {"model": "COCO yolo11n zero-shot", "precision": 0.0, "recall": 0.0, "AP50": 0.0, "mAP50-95": 0.0,
     "note": "COCO has no insulator/fault classes, so it cannot produce a single correct-class box"},
    {"model": "Trivial: predict nothing", "precision": 0.0, "recall": 0.0, "AP50": 0.0, "mAP50-95": 0.0,
     "note": "no boxes -> recall 0, AP 0"},
]
(ART / "metrics.json").write_text(json.dumps(metrics, indent=2))

## 8. From boxes to one score per photo

The business decision is made **per photo**, not per box. So for each test photo:

- **ground truth `y`** = 1 if the photo contains at least one `defect`, `brokeninsulator` or `brokenpart` label, else 0
- **score** = the **highest confidence** the model gives to any fault box in that photo. No fault box → score 0.

We run the model at a very low confidence (0.001) so we keep *every* candidate box. The thresholds are applied later, in the cost sweep. We do the same for the COCO baseline.

In [ ]:
COSTS = decision.load_costs()
POS = COSTS["positive_classes"]
assert all(p in NAMES for p in POS), f"{POS} must be in {NAMES}"
POS_IDX = [NAMES.index(p) for p in POS]

def max_conf_per_class(mdl, imgs, n_cls):
    out = np.zeros((len(imgs), n_cls))
    for start in range(0, len(imgs), 64):
        chunk = [str(p) for p in imgs[start:start + 64]]
        for j, r in enumerate(mdl.predict(chunk, conf=0.001, imgsz=IMGSZ, device=DEVICE, verbose=False, stream=True)):
            if r.boxes is None or len(r.boxes) == 0:
                continue
            cls, conf = r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()
            for c in range(n_cls):
                if np.any(cls == c):
                    out[start + j, c] = conf[cls == c].max()
    return out

test_imgs = images_in("test")
labels = [read_labels(p) for p in test_imgs]
scores = pd.DataFrame({"image": [p.name for p in test_imgs]})
for k, c in enumerate(NAMES):
    scores[f"gt_{c}"] = [int(np.sum(l[:, 0] == k)) if len(l) else 0 for l in labels]
scores["y"] = (scores[[f"gt_{NAMES[k]}" for k in POS_IDX]].sum(axis=1) > 0).astype(int)

ft = max_conf_per_class(YOLO(str(WEIGHTS / "best.pt")), test_imgs, len(NAMES))
for k, c in enumerate(NAMES):
    scores[f"score_{c}"] = ft[:, k]
scores["score"] = scores[[f"score_{NAMES[k]}" for k in POS_IDX]].max(axis=1)
scores["coco_any_score"] = max_conf_per_class(YOLO("yolo11n.pt"), test_imgs, 80).max(axis=1)
scores["trivial_score"] = 0.0

scores.to_csv(ART / "test_scores.csv", index=False)
print(f"{len(scores)} test photos, {scores['y'].mean():.0%} contain a fault")

## 9. Cost sweep → choose the thresholds

This is the core of the assignment. The maths lives in `decision.py`, so the dashboard uses exactly the same functions. Here's what it does.

### 9a. Your assumptions (`costs.json`)
| key | meaning |
|---|---|
| `cost_fn_eur` | € lost when a faulty insulator is passed |
| `cost_fp_eur` | € of a crew trip for nothing |
| `cost_tp_eur` | € of a crew trip that does fix a fault (you pay the trip either way) |
| `base_rate` | share of insulators on a **real line** that are faulty |
| `cases_per_day`, `review_seconds`, `reviewer_hourly_eur`, `reviewer_productive_hours`, `reviewers_available` | the human-review set-up |

### 9b. Why the base rate matters
The test split is mostly faulty photos, because the dataset was collected to show faults. A real line is maybe 1–3% faulty. So we measure rates **within each true class** on the test set:
- among faulty photos: what share are passed / reviewed / dispatched
- among healthy photos: the same

and then weight them by the real-world base rate π:

$$\text{cost per photo} = \pi \cdot \text{(avg cost on a faulty photo)} + (1-\pi)\cdot\text{(avg cost on a healthy photo)}$$

### 9c. Model alone (one threshold *t*)
Dispatch if score ≥ *t*, otherwise pass.
- faulty photo: dispatched → `cost_tp`; passed → `cost_fn`
- healthy photo: dispatched → `cost_fp`; passed → `cost_tn` (0)

We try *t* = 0.00, 0.01, …, 1.00, compute the cost of each, and **take the minimum**. That's the "sweep".

### 9d. Three outcomes (two thresholds *t_low* ≤ *t_high*)
- score < *t_low* → PASS
- *t_low* ≤ score < *t_high* → REVIEW: an engineer looks at it. This costs `review_seconds × hourly rate`, and we assume the engineer gets it right, so a reviewed fault becomes a correct dispatch and a reviewed healthy photo becomes a correct pass.
- score ≥ *t_high* → DISPATCH

We try every pair (*t_low*, *t_high*) on the grid and keep the cheapest one **whose review workload fits the reviewers you actually have** (`reviewers_available`):

$$\text{reviewers needed} = \frac{\text{cases/day} \times \text{review share} \times \text{review seconds}/3600}{\text{productive hours per reviewer}}$$

### 9e. Compared with three alternatives
- **Do nothing:** every insulator is passed, so every fault is missed → π × `cost_fn`
- **Engineer reviews every photo:** no misses, but you pay review time on 100% of photos
- **Model alone:** best single threshold, no humans

In [ ]:
COSTS = decision.load_costs()   # re-read, so you can edit costs.json while training runs
y, s = scores["y"].to_numpy(), scores["score"].to_numpy()

sweep = decision.sweep_single(s, y, COSTS)
sweep.to_csv(ART / "cost_sweep.csv", index=False)
best_t = float(sweep.loc[sweep["cost_per_case"].idxmin(), "threshold"])
single = decision.expected_cost(s, y, COSTS, best_t, best_t)

band_free = decision.optimise_band(s, y, COSTS)
band = decision.optimise_band(s, y, COSTS, max_reviewers=COSTS["reviewers_available"])
strategies = decision.reference_strategies(s, y, COSTS, band, single)
strategies.to_csv(ART / "strategies.csv", index=False)

print(f"Model alone: best threshold t={best_t:.2f}  ->  €{single['cost_per_1000']:.0f} per 1,000 photos")
print(f"With review: t_low={band['t_low']:.2f}, t_high={band['t_high']:.2f}  ->  €{band['cost_per_1000']:.0f} per 1,000, "
      f"{band['review_share']:.1%} reviewed, {band['reviewers_needed']:.2f} reviewers")
print(); print(strategies.round(2).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sweep["threshold"], sweep["cost_per_1000"], label="model alone")
ax.scatter([best_t], [single["cost_per_1000"]], c="red", zorder=5, label=f"minimum t={best_t:.2f}")
ax.axvspan(band["t_low"], band["t_high"], color="orange", alpha=0.2, label="chosen review band")
ax.set_xlabel("threshold on max fault score"); ax.set_ylabel("expected € per 1,000 photos")
ax.set_title("Cost against threshold"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(ART / "cost_curve.png", dpi=120)
plt.show()

### Do the baselines lose on the decision too?
For each model we run the same sweep and take its best cost. An image-level **AUC** of 0.5 means the score can't tell faulty from healthy photos at all; 1.0 means it ranks them perfectly.

In [ ]:
base = []
for name, col in (("Finetuned yolo11n (max fault score)", "score"),
                  ("COCO yolo11n zero-shot (max any-class score)", "coco_any_score"),
                  ("Trivial: predict nothing", "trivial_score")):
    sc = scores[col].to_numpy()
    sw = decision.sweep_single(sc, y, COSTS)
    b = sw.loc[sw["cost_per_case"].idxmin()]
    base.append({"model": name, "image_level_AUC": decision.roc_auc(sc, y), "best_threshold": float(b["threshold"]),
                 "best_cost_per_1000": float(b["cost_per_1000"]), "recall_at_best": float(b["test_recall"]),
                 "precision_at_best": float(b["test_precision"])})
base = pd.DataFrame(base)
base.to_csv(ART / "baselines_decision.csv", index=False)
print(base.round(3).to_string(index=False))

## 10. Save the operating point

`operating_point.json` holds the chosen thresholds, the costs they were computed with, and the training settings (for the "where trained / settings" part of REPORT.md). The dashboard loads this file. It never retrains anything.

In [ ]:
op = {"single_threshold": single, "band_unconstrained": band_free, "band_chosen": band, "costs_used": COSTS,
      "test_prevalence": float(y.mean()), "n_test_images": int(len(y)),
      "train_settings": {"model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "seed": SEED,
                         "device": DEVICE, "machine": platform.platform(), "cpu_count": os.cpu_count(),
                         "torch": torch.__version__, "minutes": round((time.time() - T0) / 60, 1)}}
(ART / "operating_point.json").write_text(json.dumps(op, indent=2, default=float))
print(f"[done] {op['train_settings']['minutes']} min. Commit weights/ and artifacts/.")